# 04 - Insights & Explainability

Which signals drive the model's fraud decisions, what does it still miss, and what does this mean in money?

In [ ]:
import sys
from pathlib import Path

# make `src` importable whether the notebook runs from /notebooks or the repo root
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import confusion_matrix

from src.config import FIG_DIR, FALSE_ALARM_COST
from src.data_prep import load_splits
from src.model import load_model, evaluate_on_test, split_xy

FIG_DIR.mkdir(parents=True, exist_ok=True)

model, meta = load_model()
_, _, test = load_splits()
feature_cols = meta["feature_columns"]
X_te, y_te, amt_te = split_xy(test, feature_cols)

print("Model:", meta["model_name"], "| threshold:", round(meta["threshold"], 3))
final = evaluate_on_test(model, meta["threshold"], test, feature_cols)

## 1. Business impact on the test set

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_te, final["pred"]).ravel()
fraud_total = amt_te[y_te == 1].sum()
caught_value = amt_te[(y_te == 1) & (final["pred"] == 1)].sum()
missed_value = amt_te[(y_te == 1) & (final["pred"] == 0)].sum()

print(f"Frauds in test set:        {tp + fn}   (total value {fraud_total:,.0f})")
print(f"Caught:                    {tp}   (value {caught_value:,.0f}, {caught_value / fraud_total:.1%} of fraud value)")
print(f"Missed:                    {fn}   (value {missed_value:,.0f})")
print(f"False alarms:              {fp:,}   (cost {fp * FALSE_ALARM_COST:,.0f} at {FALSE_ALARM_COST}/alarm)")
print(f"Total cost with model:     {final['cost']:,.0f}")
print(f"Total cost doing nothing:  {final['baselines']['no_model']:,.0f}")
print(f"Reduction:                 {final['savings_vs_no_model']:.1%}")

## 2. What drives the predictions? (SHAP)

SHAP values attribute each prediction to the input features. I explain the model on all test frauds plus a random sample of legitimate
transactions (a plain random sample would contain almost no fraud to learn from).

In [ ]:
# SHAP explains the final estimator, so push data through any preprocessing steps first
# (scaler for logistic regression). SMOTE is training-only and is skipped at predict time.
final_est = model.steps[-1][1]
X_explain_all = pd.concat([
    X_te[y_te == 1],
    X_te[y_te == 0].sample(2000, random_state=0),
])
if len(model.steps) > 1:
    pre = model[:-1]
    X_explain = pd.DataFrame(pre.transform(X_explain_all), columns=X_explain_all.columns, index=X_explain_all.index)
else:
    X_explain = X_explain_all

explainer = shap.Explainer(final_est, X_explain)
shap_values = explainer(X_explain)
shap_values.values.shape

In [ ]:
shap.summary_plot(shap_values, X_explain, show=False, max_display=15)
plt.title("SHAP summary - top drivers of the fraud score")
plt.savefig(FIG_DIR / "04_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
shap.plots.bar(shap_values, max_display=15, show=False)
plt.savefig(FIG_DIR / "04_shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()

**How to read the summary plot:** each dot is one transaction; position on the x-axis shows whether the feature pushed the score towards fraud (right) or legitimate (left);
colour shows the feature's value. Features at the top matter most overall.

**My own observations:** _(which 3 features dominate? do they match the separation chart from `01_eda`?)_

## 3. Explaining individual decisions

In [ ]:
# the transactions the model was MOST sure were fraud, and truly were
fraud_idx = [i for i, ix in enumerate(X_explain.index) if y_te.loc[ix] == 1]
scores_explain = model.predict_proba(X_explain_all)[:, 1]
top = sorted(fraud_idx, key=lambda i: -scores_explain[i])[:1]

for i in top:
    print(f"Fraud score: {scores_explain[i]:.3f}  |  amount: {X_explain_all.iloc[i]['Amount']:.2f}")
    shap.plots.waterfall(shap_values[i], max_display=10, show=False)
    plt.savefig(FIG_DIR / "04_waterfall_caught_fraud.png", dpi=150, bbox_inches="tight")
    plt.show()

## 4. What does the model still miss?

In [ ]:
missed = test.loc[(y_te == 1) & (final["pred"] == 0)].copy()
missed["fraud_score"] = final["scores"][(y_te == 1).values & (final["pred"] == 0)]
print(f"{len(missed)} missed frauds. Amounts and scores:")
missed[["Amount", "fraud_score"]].sort_values("Amount", ascending=False).round(3).head(10)

In [ ]:
caught = test.loc[(y_te == 1) & (final["pred"] == 1)]
print(f"Median amount - caught frauds: {caught['Amount'].median():.2f}  |  missed frauds: {missed['Amount'].median():.2f}")

Missed frauds are usually the ones that look statistically ordinary - they sit close to legitimate transactions in feature space.
Whether the *costly* ones are being missed (large amounts) or only small ones tells the business whether to adjust the threshold.

## 5. Limitations & next steps

- **Anonymised features:** `V1`-`V28` are PCA components, so I can rank them but not translate them into business rules.
- **Two days of data:** no seasonality, and fraud patterns drift - a production system needs monitoring and periodic retraining.
- **Random (stratified) split:** a time-based split would be a stricter test of how the model ages.
- **Cost assumptions:** the false-alarm cost is a guess; `03_modeling` shows the decision is sensitive to it.
- **Possible extensions:** hyper-parameter tuning, probability calibration, an isolation-forest / autoencoder anomaly detector as a comparison, a time-based validation scheme.